# Refinamento — Selo Ambiental 2026 (SEMARH-PI)

Transforma o dado **bruto** do Selo Ambiental num dataset **pronto para o painel**.

| | |
|---|---|
| **Origem** | `s3a://entrada/sermarh_painel/selo_ambiental_2026.parquet` (a mesma que o Dremio expõe em `coleta."semarh_painel"`) |
| **Destino** | `nessie.refinamento.semarh_painel` (+ dois resumos) |
| **Grão** | 1 linha por município (os 224 do Piauí) |

**Rodadas 1/2/3** são as fases de avaliação/recurso da edição 2026. A **rodada 3 é o resultado
final** (consolidado), então é dela que saem `resultado`, `pontos` e `criterios_atendidos`.

O `resultado` de cada município é um de: **Selo A/B/C**, **Não elegível** ou **Não habilitado**.
A coluna derivada `situacao` agrupa isso em: *Com selo*, *Não elegível*, *Não habilitado* e
*Não postulado* (esta fica em 0 nesta edição — todos os 224 municípios postularam).


## 1. Ler o dado bruto (camada de entrada, no MinIO)


In [ ]:
from pyspark.sql import functions as F
from lakehouse import sessao, ler_arquivo, gravar, perfil, listar

spark = sessao("refinamento-selo-ambiental")

# ler_arquivo usa a zona de entrada por padrao -> s3a://entrada/<caminho>
bruto = ler_arquivo(spark, "sermarh_painel/selo_ambiental_2026.parquet")
print("linhas brutas:", bruto.count())
bruto.printSchema()

## 2. Refinar

Renomeia para nomes limpos (sem acento/espaço), tipa, deriva `selo`, `situacao`,
`pacto_ambiental` e extrai `latitude`/`longitude` do campo de coordenadas.


In [ ]:
# RESULTADO/PONTOS/CRITERIOS vem da rodada 3 (resultado final).
res   = F.upper(F.trim(F.col("`RESULTADO 3`")))
# aux3 = "lat, lon"  ->  remove espacos e divide na virgula
coord = F.split(F.regexp_replace(F.col("aux3"), " ", ""), ",")

ref = (
    bruto.select(
        F.trim(F.col("`MUNICÍPIO`")).alias("municipio"),
        F.trim(F.col("PROCESSO")).alias("processo"),
        F.trim(F.col("aux2")).alias("territorio_desenvolvimento"),
        (F.upper(F.trim(F.col("`HABILITADO 3`"))) == "SIM").alias("habilitado"),
        F.col("`PONTOS 3`").cast("double").alias("pontos"),
        F.col("`CRITÉRIOS 3`").cast("int").alias("criterios_atendidos"),
        res.alias("_res"),
        (F.lower(F.trim(F.col("pactos"))) == "sim").alias("pacto_ambiental"),
        coord.getItem(0).cast("double").alias("latitude"),
        coord.getItem(1).cast("double").alias("longitude"),
    )
    .withColumn("resultado",
        F.when(F.col("_res") == "SELO A", "Selo A")
         .when(F.col("_res") == "SELO B", "Selo B")
         .when(F.col("_res") == "SELO C", "Selo C")
         .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
         .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado"))
    .withColumn("selo", F.regexp_extract(F.col("_res"), r"SELO ([ABC])", 1))
    .withColumn("selo", F.when(F.col("selo") == "", None).otherwise(F.col("selo")))
    .withColumn("tem_selo", F.col("_res").startswith("SELO"))
    .withColumn("situacao",
        F.when(F.col("_res").startswith("SELO"), "Com selo")
         .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
         .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado")
         .otherwise("Não postulado"))
    .drop("_res")
    .select(
        "municipio", "processo", "territorio_desenvolvimento", "situacao",
        "resultado", "selo", "tem_selo", "pontos", "criterios_atendidos",
        "habilitado", "pacto_ambiental", "latitude", "longitude",
    )
)
perfil(ref)

## 3. Validar antes de gravar


In [ ]:
regras = {
    "224 municípios":        ref.count() == 224,
    "município único":       ref.select("municipio").distinct().count() == ref.count(),
    "toda linha com coord":  ref.filter("latitude IS NULL OR longitude IS NULL").count() == 0,
    "situação preenchida":   ref.filter("situacao IS NULL").count() == 0,
    "selo só quando tem":    ref.filter("tem_selo = true AND selo IS NULL").count() == 0,
}
for regra, passou in regras.items():
    print(f"  {'OK  ' if passou else 'FALHOU'}  {regra}")
assert all(regras.values()), "corrija antes de gravar"

## 4. Gravar a tabela principal em `refinamento`


In [ ]:
gravar(ref, "refinamento.semarh_painel", modo="substituir")

## 5. Resumos para o painel

Dois recortes que o dashboard consome direto: municípios **por tipo de selo/situação** e
**por número de critérios atendidos**.


In [ ]:
por_selo = (
    ref.groupBy("resultado")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("resultado")
)
por_criterios = (
    ref.groupBy("criterios_atendidos")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("criterios_atendidos")
)
gravar(por_selo,      "refinamento.semarh_painel_por_selo",      modo="substituir")
gravar(por_criterios, "refinamento.semarh_painel_por_criterios", modo="substituir")

## 6. Conferir o resultado


In [ ]:
listar(spark, "refinamento")
print("\nMunicípios por situação:")
ref.groupBy("situacao").count().orderBy(F.desc("count")).show(truncate=False)
print("Municípios por tipo de selo (resultado final):")
por_selo.show(truncate=False)
print("Pontuação e resultado (amostra, maiores pontuações):")
ref.select("municipio", "resultado", "pontos").orderBy(F.desc("pontos")).show(10, truncate=False)